# Tuần 2 — PhoBERT Multi-task Finetune (Anti-OOM)
**ABSA VLSP 2018 Hotel | NLP Course — HUST**

Notebook này dựa trên `week2_phobert_training_version1.ipynb` và chỉ sửa cấu hình để hạn chế OOM CUDA:
- `max_seq_len = 256`
- `batch_size = 4`
- `grad_accumulation_steps = 8`

> Chạy theo thứ tự từ Cell 1 đến Cell 9.

In [1]:
# ============================================================
# Cell 1 — Check GPU & Install dependencies
# ============================================================
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu}")
    print(f"   VRAM: {vram:.1f} GB")
    if vram < 14:
        print("⚠️  VRAM < 14GB — nếu OOM thì giảm batch_size về 4 trong constants.py")
else:
    raise RuntimeError("❌ Không có GPU! Kaggle: Settings → Accelerator → GPU T4 x2")

torch.cuda.empty_cache()
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

# Install packages
!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece
print("✅ Dependencies installed")

✅ GPU: Tesla T4
   VRAM: 15.6 GB
PyTorch: 2.10.0+cu128 | CUDA: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 76.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 93.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 98.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 73.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.
✅ Dependencies installed


In [2]:
# ============================================================
# Cell 2 — Clone repo từ GitHub & Setup working directory
# ============================================================
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "master"
PROJECT_DIR = "/kaggle/working/absa-project"

# Clone (hoặc pull nếu đã có)
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print("✅ Clone xong")
else:
    print(f"Repo đã tồn tại tại {PROJECT_DIR} — pulling latest...")
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

# Chuyển vào thư mục project
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

# Tạo thư mục cần thiết
for d in ["data", "outputs/models", "outputs/results", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

# Thêm Python paths
sys.path.insert(0, "code/week1")
sys.path.insert(0, "code/week2")

# Kiểm tra cấu trúc repo
print("\nCấu trúc repo:")
!ls -la
!ls code/week1/ code/week2/

Cloning https://github.com/vudinhminh08/NLP-project-master-study.git (branch: master)...
Cloning into '/kaggle/working/absa-project'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 87 (delta 9), reused 75 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 3.21 MiB | 20.54 MiB/s, done.
Resolving deltas: 100% (9/9), done.
✅ Clone xong
Working dir: /kaggle/working/absa-project

Cấu trúc repo:
total 52
drwxr-xr-x 8 root root  4096 Mar 29 18:21  .
drwxr-xr-x 4 root root  4096 Mar 29 18:21  ..
drwxr-xr-x 6 root root  4096 Mar 29 18:21  code
drwxr-xr-x 2 root root  4096 Mar 29 18:21  data
-rw-r--r-- 1 root root 10244 Mar 29 18:21  .DS_Store
drwxr-xr-x 8 root root  4096 Mar 29 18:21  .git
drwxr-xr-x 2 root root  4096 Mar 29 18:21  notebooks
drwxr-xr-x 5 root root  4096 Mar 29 18:21  outputs
-rw-r--r-- 1 root root  1238 Mar 29 18:21  README.md
-rw-r--r-- 1 root root   372 Mar 2

In [3]:
# ============================================================
# Cell 3 — Download dataset VLSP 2018
# ============================================================
import pandas as pd, os

if not os.path.exists("data/train.csv"):
    print("Downloading VLSP 2018 Hotel dataset...")
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print("✅ Data downloaded")
else:
    print("✅ Data đã tồn tại")

for split in ["train", "dev", "test"]:
    df = pd.read_csv(f"data/{split}.csv")
    print(f"  {split}: {len(df)} rows × {df.shape[1]} cols")

✅ Data đã tồn tại
  train: 3000 rows × 35 cols
  dev: 2000 rows × 35 cols
  test: 600 rows × 35 cols


In [4]:
# ============================================================
# Cell 4 — Preprocessing
# ============================================================
import pandas as pd, os

FORCE_REPROCESS = False

if (not FORCE_REPROCESS) and os.path.exists("data/train_preprocessed.csv"):
    print("✅ Cache đã có (data/*_preprocessed.csv)")
    s = pd.read_csv("data/train_preprocessed.csv").iloc[0]
    print(f"  Original : {s['Review'][:80]}")
    print(f"  Processed: {str(s.get('processed_review', 'N/A'))[:80]}")
else:
    print("Chưa có cache — chạy preprocessing với VnCoreNLP...")
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    import py_vncorenlp
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models', 'wordsegmenter', 'wordsegmenter.rdr')):
        print('Downloading VnCoreNLP models...')
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f"data/{split}_preprocessed.csv")
        print(f"  ✅ {split}: {len(df)} rows processed")
    segmenter.close()
    print("✅ Preprocessing hoàn tất")

✅ Cache đã có (data/*_preprocessed.csv)
  Original : Rộng rãi KS mới nhưng rất vắng. Các dịch vụ chất lượng chưa cao và thiếu.
  Processed: Rộng_rãi_khách_sạn_mới_nhưng_rất_vắng_._Các_dịch_vụ_chất_lượng_chưa_cao_và_thiếu


In [ ]:
# ============================================================
# Cell 5 — Verify config (ds4v sync, không cần override thủ công)
# ============================================================
import json, os

enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
print('=== Encoder Config ===')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

cw = json.load(open('outputs/eda/class_weights.json'))
print('\n=== Global Class Weights ===')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    note = ' ← clip về 10.0' if float(w) > 10 else ''
    print(f"  {label_map.get(cls,cls):12s}: {float(w):.1f}x{note}")

from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, PHOBERT_MODEL_NAME

print('\n=== Train Config (ds4v sync) ===')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')
print(f'  model: {PHOBERT_MODEL_NAME}')

# Verify ds4v values
assert TRAIN_CONFIG['learning_rate'] == 1e-4, f"LR sai: {TRAIN_CONFIG['learning_rate']}"
assert TRAIN_CONFIG['optimizer'] == 'Adam', f"Optimizer sai: {TRAIN_CONFIG['optimizer']}"
assert PHOBERT_MODEL_NAME == 'vinai/phobert-base-v2', f"Model sai: {PHOBERT_MODEL_NAME}"
print(f'\n  ZERO_TRAIN_ASPECTS: {ZERO_TRAIN_ASPECTS}')
print('\n✅ Config OK — ds4v sync verified')


In [6]:
# ============================================================
# Cell 6 — TRAIN: concat_4_layers (config anti-OOM đã set ở Cell 5)
# ============================================================
import torch
torch.cuda.empty_cache()
print(f"VRAM free before training: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

from run_experiment import main

test_metrics = main(encoder_option='concat_4_layers', use_amp=True)

print('\n' + '='*55)
print('MAIN RUN RESULTS')
print(f"  ACD F1:      {test_metrics['macro_acd_f1']:.4f}  (SOTA: 0.8255)")
print(f"  SPC F1:      {test_metrics['macro_spc_f1']:.4f}")
print(f"  Combined F1: {test_metrics['macro_combined_f1']:.4f}  (SOTA: 0.7732)")
print('='*55)


VRAM free before training: 15.6 GB
[Device] GPU: Tesla T4

TUẦN 2 — PhoBERT Multi-task ABSA
  encoder:    concat_4_layers
  seq_len:    256
  batch:      8 × 2 = 16 effective
  lr:         2e-05
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...


  Dev — Epoch 19
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    7
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    36
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
HOTEL#MISCELLANEOUS                  0.0134    0.2500    0.0069    0.0065    145
ROOMS#QUALITY                        0.0588    0.0357    0.1667    0.1667    6
ROOM_AMENITIES#COMFORT               0.0684    0.4500    0.0370    0.0319    243
FOOD&DRINKS#PRICES                   0.0870    0.1176    0.0690    0.0905    29
FACILITIES#CO

Train:   0%|          | 0/375 [00:00<?, ?it/s]/kaggle/working/absa-project/code/week2/train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  Dev — Epoch 20
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    7
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    36
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
HOTEL#MISCELLANEOUS                  0.0134    0.2500    0.0069    0.0065    145
ROOMS#QUALITY                        0.0571    0.0345    0.1667    0.1667    6
FOOD&DRINKS#PRICES                   0.0851    0.1111    0.0690    0.0905    29
ROOM_AMENITIES#COMFORT               0.0899    0.5000    0.0494    0.0524    243
FACILITIES#CO


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    7
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    36
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
HOTEL#MISCELLANEOUS                  0.0134    0.2500    0.0069    0.0065    145
ROOMS#QUALITY                        0.0571    0.0345    0.1667    0.1667    6
FOOD&DRINKS#PRICES                   0.0851    0.1111    0.0690    0.0905    29
ROOM_AMENITIES#COMFORT               0.0899    0.5000    0.0494    0.0524    243
FACILITIES#COMFO


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    13
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    3
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.0278    0.2500    0.0147    0.0000    68
FACILITIES#COMFORT                   0.0690    0.3333    0.0385    0.0317    26
FOOD&DRINKS#PRICES                   0.1176    0.1250    0.1111    0.0000    9
FACILITIES#CLEANLINESS               0.1333    0.1000    0.2000    0.0000    5
FACILITIES#QUALITY 

In [7]:
# ============================================================
# Cell 7 — Ablation: cls_only
# ============================================================
# Chứng minh kỹ thuật concat 4 layers có đóng góp thực sự
# (so sánh 3072 dim vs 768 dim trong báo cáo)
# Chạy SAU khi Cell 6 đã hoàn tất

import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics_cls = main(encoder_option="cls_only", use_amp=True)

print("\n" + "="*55)
print("ABLATION SUMMARY")
print(f"  concat_4_layers: {test_metrics['macro_combined_f1']:.4f}  ← SOTA architecture")
print(f"  cls_only:        {test_metrics_cls['macro_combined_f1']:.4f}")
gain = test_metrics['macro_combined_f1'] - test_metrics_cls['macro_combined_f1']
print(f"  Gain từ concat:  {gain*100:+.2f}%  {'✅ concat tốt hơn' if gain > 0 else '⚠️ cls_only bằng hoặc tốt hơn'}")
print("="*55)

[Device] GPU: Tesla T4

TUẦN 2 — PhoBERT Multi-task ABSA
  encoder:    cls_only
  seq_len:    256
  batch:      8 × 2 = 16 effective
  lr:         2e-05
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Tokenizer] Loaded ✓

[Data] Creating DataLoaders...
[DataLoader] train: 3000 samples, 375 batches
[DataLoader] dev: 2000 samples, 250 batches
[DataLoader] test: 600 samples, 75 batches
[Weights] 34 aspects loaded, 77 weight values clipped at 10.0

[Model] Building ABSAPhoBERT (cls_only)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/kaggle/working/absa-project/code/week2/train.py:253: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (use_amp and AMP_AVAILABLE and device.type == "cuda") else None



[Model] 135,102,856 trainable parameters
[Scheduler] Total=3760 optimizer steps, Warmup=376
[AMP] Mixed precision: ON ✓
[Config] encoder=cls_only, seq_len=256, batch=8×2=16 (effective)

────────────────────────────────────────────────────────────
Epoch 1/20


...


  Dev — Epoch 16
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    7
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    36
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
HOTEL#MISCELLANEOUS                  0.0000    0.0000    0.0000    0.0000    145
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
FACILITIES#COMFORT                   0.0282    0.5000    0.0145    0.0000    69
FOOD&DRINKS#PRICES                   0.0571    0.1667    0.0345    0.0392    29
* ROOM_AMENITIES#CLEANLINESS         0.0847    0.4167    0.0472    0.0325    106
ROOM_AMENITI

Train:   0%|          | 0/375 [00:00<?, ?it/s]/kaggle/working/absa-project/code/week2/train.py:139: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  Dev — Epoch 17
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    7
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    36
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
HOTEL#MISCELLANEOUS                  0.0000    0.0000    0.0000    0.0000    145
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
FACILITIES#COMFORT                   0.0286    1.0000    0.0145    0.0000    69
FOOD&DRINKS#PRICES                   0.0588    0.2000    0.0345    0.0392    29
ROOM_AMENITIES#COMFORT               0.0692    0.5294    0.0370    0.0357    243
* ROOM_AMENI


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    7
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    36
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
HOTEL#MISCELLANEOUS                  0.0000    0.0000    0.0000    0.0000    145
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
FOOD&DRINKS#PRICES                   0.0488    0.0833    0.0345    0.0392    29
ROOMS#QUALITY                        0.0667    0.0417    0.1667    0.1667    6
ROOM_AMENITIES#COMFORT               0.0687    0.4737    0.0370    0.0317    243
FACILITIES#COMFO


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
FACILITIES#COMFORT                   0.0000    0.0000    0.0000    0.0000    26
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
FACILITIES#PRICES                    0.0000    0.0000    0.0000    0.0000    13
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    3
FOOD&DRINKS#PRICES                   0.0000    0.0000    0.0000    0.0000    9
HOTEL#MISCELLANEOUS                  0.0000    0.0000    0.0000    0.0000    68
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
FACILITIES#QUALITY                   0.1017    0.3750    0.0588    0.0238    51
FACILITIES#CLEANLI

In [9]:
# ============================================================
# Cell 8 — Learning Curve (BẮT BUỘC cho báo cáo)
# ============================================================
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open("outputs/results/training_history.json"))
best_ep = history["best_epoch"]
epochs  = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PhoBERT concat_4_layers — Learning Curve (ABSA VLSP 2018)", fontsize=13)

# --- Loss ---
ax1.plot(epochs, history["train_loss"], "o-", c="crimson",   lw=2, label="Train Loss")
ax1.plot(epochs, history["dev_loss"],   "o-", c="steelblue", lw=2, label="Dev Loss")
ax1.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax1.set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
ax1.legend(); ax1.grid(alpha=0.3)

# --- F1 ---
ax2.plot(epochs, history["dev_acd_f1"],      "s-", c="darkorange", lw=2, label="Dev ACD F1")
ax2.plot(epochs, history["dev_spc_f1"],      "^-", c="purple",     lw=2, label="Dev SPC F1")
ax2.plot(epochs, history["dev_combined_f1"], "o-", c="green",      lw=2.5, label="Dev Combined F1")
ax2.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax2.axhline(0.7732,  c="red",   ls=":",  alpha=0.5, label="SOTA Combined 0.7732")
ax2.set(title="F1 Score", xlabel="Epoch", ylabel="Macro F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Phân tích cho báo cáo ---
print(f"\n📊 Phân tích learning curve (cho báo cáo):")
print(f"  Best epoch:           {best_ep} / {len(history['train_loss'])}")
print(f"  Best Combined F1:     {history['best_combined_f1']:.4f}")

if len(history["train_loss"]) > best_ep:
    tloss_at   = history["train_loss"][best_ep - 1]
    tloss_after = history["train_loss"][best_ep]
    dloss_at   = history["dev_loss"][best_ep - 1]
    dloss_after = history["dev_loss"][best_ep]
    print(f"  Train loss epoch {best_ep}: {tloss_at:.4f} → epoch {best_ep+1}: {tloss_after:.4f} (tiếp tục giảm)")
    print(f"  Dev loss epoch {best_ep}:   {dloss_at:.4f} → epoch {best_ep+1}: {dloss_after:.4f} (tăng = overfit)")
    print(f"  → Early stopping đúng: dev F1 không cải thiện {history['config']['early_stop_patience']} epoch liên tiếp")

print(f"\n  Saved: outputs/eda/learning_curve.png")


📊 Phân tích learning curve (cho báo cáo):
  Best epoch:           20 / 20
  Best Combined F1:     0.2821

  Saved: outputs/eda/learning_curve.png


In [10]:
# ============================================================
# Cell 9 — Summary Report & Copy kết quả
# ============================================================
import os, json, shutil

# In summary report
report = "outputs/results/week2_summary.md"
if os.path.exists(report):
    print(open(report, encoding="utf-8").read())
else:
    print("Chưa có report — đảm bảo Cell 6 đã chạy xong")

# Liệt kê tất cả files kết quả
print("\n=== Kết quả đã tạo ===")
result_files = []
for root, dirs, files in os.walk("outputs"):
    for f in files:
        if not f.endswith(".DS_Store"):
            path = os.path.join(root, f)
            size = os.path.getsize(path)
            result_files.append(path)
            print(f"  {path} ({size/1024:.1f} KB)")

# Tạo zip để download
print("\nTạo zip...")
shutil.make_archive("/kaggle/working/week2_results", "zip", "outputs")
print("✅ Zip: /kaggle/working/week2_results.zip")
print("   → Kaggle: Output panel → Download")

# Copy summary để dễ copy-paste
print("\n=== Số liệu quan trọng để điền vào SESSION_HANDOFF.md ===")
if os.path.exists("outputs/results/week2_test_metrics.json"):
    m = json.load(open("outputs/results/week2_test_metrics.json"))
    print(f"__WEEK2_ACD_F1__      = {m['macro_acd_f1']:.4f}")
    print(f"__WEEK2_SPC_F1__      = {m['macro_spc_f1']:.4f}")
    print(f"__WEEK2_COMBINED_F1__ = {m['macro_combined_f1']:.4f}")
if os.path.exists("outputs/results_cls_only/week2_test_metrics.json"):
    m2 = json.load(open("outputs/results_cls_only/week2_test_metrics.json"))
    print(f"__WEEK2_ABLATION_COMBINED_F1__ = {m2['macro_combined_f1']:.4f}")

# Kết quả Tuần 2 — PhoBERT Multi-task

## Config thực tế
| Tham số | Giá trị |
|---------|---------|
| Encoder | concat_4_layers |
| MAX_SEQ_LEN | 256 |
| Batch size | 8 × 2 = 16 (effective) |
| Learning rate | 2e-05 |
| Warmup | 10% steps |
| Weight clip | 10.0 |
| Best epoch | 20 / 20 |

## Kết quả

| Split | ACD F1 | SPC F1 | Combined F1 |
|-------|--------|--------|-------------|
| Dev   | 0.3345 | 0.2297 | 0.2821 |
| **Test**  | **0.3592** | **0.2371** | **0.2981** |
| SOTA (Huynh 2022) | 0.8255 | — | 0.7732 |

## Phân tích Gap so với SOTA

- **ACD F1 gap:** 0.4663 (46.6%)
- **Combined F1 gap:** 0.4751 (47.5%)

### Nguyên nhân gap (phân tích):
1. **underthesea vs VnCoreNLP:** Dùng underthesea làm fallback → ~1-2% F1 loss
2. **ROOM_AMENITIES#PRICES:** 0 training samples → ACD F1 = 0 cho aspect này
3. **Neutral cực hiếm (weight=154→clip=10):** SPC F1 cho neutral thấp
4. **Dataset nhỏ (3000 train):** SOTA có thể dùng data augmentation
5. **Single run:** Chưa ensemble nhiều seeds

## 